In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# Path dataset
train_dir = "D:/oulunpu/dataset/train"
val_dir = "D:/oulunpu/dataset/val"

# Data augmentation untuk training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.95, 1.05]
)

# Data preprocessing untuk validasi & testing (tanpa augmentasi)
val_datagen = ImageDataGenerator(rescale=1./255)

# Load dataset
batch_size = 32
img_size = (224, 224)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

Found 8730 images belonging to 2 classes.
Found 6544 images belonging to 2 classes.


In [3]:
# Load EfficientNetV2B0 tanpa fully connected layer
base_model = EfficientNetV2B0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Fine-tune hanya 30 layer terakhir
for layer in base_model.layers[-30:]:
    layer.trainable = True

# Tambahkan lapisan klasifikasi
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(0.0005))(x)
x = Dropout(0.4)(x)
x = Dense(64, activation='relu', kernel_regularizer=l2(0.0005))(x)
x = Dropout(0.4)(x)
x = Dense(32, activation='relu', kernel_regularizer=l2(0.0005))(x)
x = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid', kernel_regularizer=l2(0.0005))(x)

# Buat model akhir
model = Model(inputs=base_model.input, outputs=output)

# Compile model
model.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    metrics=["accuracy"]
)

# Callbacks
early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_model.h5", monitor="val_accuracy", save_best_only=True)

# Training model
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=[early_stopping, checkpoint]
)

model.summary()

Epoch 1/50
273/273 [==============================] - 393s 1s/step - loss: 0.8261 - accuracy: 0.6354 - val_loss: 0.7907 - val_accuracy: 0.7965
Epoch 2/50
273/273 [==============================] - 126s 462ms/step - loss: 0.7409 - accuracy: 0.7380 - val_loss: 0.7252 - val_accuracy: 0.8004
Epoch 3/50
273/273 [==============================] - 128s 467ms/step - loss: 0.6579 - accuracy: 0.7781 - val_loss: 0.6569 - val_accuracy: 0.8267
Epoch 4/50
273/273 [==============================] - 123s 452ms/step - loss: 0.5687 - accuracy: 0.8132 - val_loss: 0.6434 - val_accuracy: 0.8748
Epoch 5/50
273/273 [==============================] - 120s 439ms/step - loss: 0.4943 - accuracy: 0.8395 - val_loss: 0.6304 - val_accuracy: 0.8533
Epoch 6/50
273/273 [==============================] - 122s 445ms/step - loss: 0.4346 - accuracy: 0.8731 - val_loss: 0.5568 - val_accuracy: 0.8412
Epoch 7/50
273/273 [==============================] - 122s 445ms/step - loss: 0.3847 - accuracy: 0.9115 - val_loss: 0.4464 - va

In [4]:
test_dir = "D:/oulunpu/dataset/test"
test_datagen = ImageDataGenerator(rescale=1./255)

# Load dataset test (tanpa label)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode="binary",  # Model butuh label untuk evaluasi
    shuffle=False
)
# Evaluasi model
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc*100:.2f}%")


Found 8762 images belonging to 2 classes.
274/274 [==============================] - 84s 305ms/step - loss: 0.2944 - accuracy: 0.9618
Test Accuracy: 96.18%


In [5]:
model.save("face_anti_spoofing_efficientnetv2_final.h5")
